In [1]:
%load_ext autoreload
%autoreload 2  

In [2]:
import numpy as np
from src.game.connect_four import BoardState, ConnectFourGame
from src.mcts.tree import Node
from src.mcts.mcts_algorithm import MCTS

In [3]:
game = ConnectFourGame.as_new_game(verbose = True)
root_node = Node.as_root_node_from_game(
    game = game
)
print(len(root_node.untried_actions))  # should be 7
print(len(root_node.children)) 

7
0


In [4]:
game = ConnectFourGame.as_new_game(verbose = True)
seed = 0
rng = np.random.default_rng(seed=seed)

while game.result is None:
# for i in range(5):
    root_node = Node.as_root_node_from_game(game=game.copy())
    mcts = MCTS(
        root_node = root_node,
        rng = rng
    )
    optimal_action = mcts.run(n_iters = int(1e2))
    game.make_move(move = optimal_action.move)
    game.print_board()

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . X . .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X . .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X X .
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
. . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
X . . . . X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . O .
X . . . O X .
O X O . X X O
0 1 2 3 4 5 6
. . . . . . .
. . . 

In [7]:
from collections import Counter

def play_game(n_iters: int, rng: np.random.Generator) -> int:
    game = ConnectFourGame.as_new_game()
    
    while game.result is None:
        root = Node.as_root_node_from_game(game=game)
        mcts = MCTS(root_node=root, rng=rng)
        action = mcts.run(n_iters=n_iters)
        game.make_move(action.move)
    
    return game.result

rng = np.random.default_rng(42)
results = Counter()

for i in range(1000):
    result = play_game(n_iters=100, rng=rng)
    results[result] += 1
    if i % 100 == 0:
        print(f"Game {i}: {dict(results)}")

print(f"\nFinal results over 1000 games:")
print(f"X wins (1):  {results[1]}")
print(f"O wins (-1): {results[-1]}")
print(f"Draws  (0):  {results[0]}")
print(f"X win rate:  {results[1]/1000:.1%}")

Game 0: {-1: 1}
Game 100: {-1: 43, 1: 57, 0: 1}
Game 200: {-1: 84, 1: 113, 0: 4}
Game 300: {-1: 115, 1: 181, 0: 5}
Game 400: {-1: 155, 1: 240, 0: 6}
Game 500: {-1: 193, 1: 300, 0: 8}
Game 600: {-1: 228, 1: 363, 0: 10}
Game 700: {-1: 271, 1: 418, 0: 12}
Game 800: {-1: 308, 1: 478, 0: 15}
Game 900: {-1: 355, 1: 529, 0: 17}

Final results over 1000 games:
X wins (1):  584
O wins (-1): 397
Draws  (0):  19
X win rate:  58.4%


In [29]:
import numpy as np


state = np.array([
    [ 0,  0,  1, -1, -1,  0,  0],  # row 0
    [ 1, -1,  1,  1, -1,  0,  0],  # row 1
    [-1,  1, -1,  1,  1,  0,  0],  # row 2
    [ 1,  1, -1,  1, -1,  0,  0],  # row 3
    [-1, -1,  1, -1, -1, -1,  0],  # row 4  ← O in cols 0,1,3,4,5
    [ 1, -1,  1,  1, -1,  1,  1],  # row 5
], dtype=np.int8)

board = BoardState(state=state)
game = ConnectFourGame(board_state=board, whose_turn=-1, verbose = True)

In [59]:
state = np.array([
    [ 0,  0,  0,  0,  0,  0,  0],
    [ 0, -1,  0,  0,  0,  0,  1],
    [ 0, -1, -1,  0,  1,  1,  1],
    [ 0,  1, -1,  0,  1, -1,  1],
    [ 1, -1,  1,  0, -1,  1, -1],
    [-1,  1, -1, -1,  1, -1, -1],
], dtype=np.int8)

board = BoardState(state=state)
game = ConnectFourGame(board_state=board, whose_turn=1)

rng = np.random.default_rng(42)
root = Node.as_root_node_from_game(game=game)
mcts = MCTS(root_node=root, rng=rng)
action = mcts.run(n_iters=5000)

print(f"Chosen action: {action.move}")
print(f"Root: visits={root.n_visits}, wins={root.n_wins}")
for child in sorted(root.children, key=lambda c: c.parent_action.move):
    print(f"Col {child.parent_action.move}: visits={child.n_visits}, wins={child.n_wins:.1f}")

Chosen action: 6
Root: visits=5000, wins=4791
Col 0: visits=273, wins=232.0
Col 1: visits=306, wins=264.0
Col 2: visits=316, wins=274.0
Col 3: visits=1694, wins=1694.0
Col 4: visits=460, wins=417.0
Col 5: visits=256, wins=215.0
Col 6: visits=1695, wins=1695.0


In [ ]:
# rng = np.random.default_rng(42)
# root = Node.as_root_node_from_game(game=game)
# mcts = MCTS(root_node=root, rng=rng)
# action = mcts.run(n_iters=1)

In [ ]:
# rng = np.random.default_rng(42)
# root = Node.as_root_node_from_game(game=game)
# mcts = MCTS(root_node=root, rng=rng)
# action = mcts.run(n_iters=1)

# print(f"Root: visits={root.n_visits}, wins={root.n_wins}")
# print(f"Total wins across all children: {sum(c.n_wins for c in root.children)}")
# for child in sorted(root.children, key=lambda c: c.parent_action.move):
#     print(f"Col {child.parent_action.move}: visits={child.n_visits}, wins={child.n_wins:.1f}")

-1
[]
[Action(move=0, prior=0.0), Action(move=1, prior=0.0), Action(move=5, prior=0.0), Action(move=6, prior=0.0)]
[Node(id=0, game=ConnectFourGame(board_state=BoardState(state=array([[ 0,  0,  1, -1, -1,  0,  0],
       [ 1, -1,  1,  1, -1,  0,  0],
       [-1,  1, -1,  1,  1,  0,  0],
       [ 1,  1, -1,  1, -1,  0,  0],
       [-1, -1,  1, -1, -1, -1,  0],
       [ 1, -1,  1,  1, -1,  1,  1]], dtype=int8)), whose_turn=-1, result=None, verbose=True), n_visits=0, n_wins=0, untried_actions=[Action(move=0, prior=0.0), Action(move=1, prior=0.0), Action(move=5, prior=0.0), Action(move=6, prior=0.0)], parent=None, parent_action=None, children=[])]
-1
Node(id=696996, game=ConnectFourGame(board_state=BoardState(state=array([[ 0,  0,  1, -1, -1,  0,  0],
       [ 1, -1,  1,  1, -1,  0,  0],
       [-1,  1, -1,  1,  1,  0,  0],
       [ 1,  1, -1,  1, -1,  0,  0],
       [-1, -1,  1, -1, -1, -1, -1],
       [ 1, -1,  1,  1, -1,  1,  1]], dtype=int8)), whose_turn=-1, result=-1, verbose=False), 

In [26]:
print(f"Total wins across all children: {sum(c.n_wins for c in root.children)}")
print(f"Root wins: {root.n_wins}")

Total wins across all children: 4385.0
Root wins: 5615.0


In [15]:
print(f"Root: visits={root.n_visits}, wins={root.n_wins}")

Root: visits=10000, wins=5605.5


In [13]:
game = ConnectFourGame(board_state=board, whose_turn=-1)
print(game.result)

None


In [5]:
game = ConnectFourGame.as_new_game()
root_node = Node.as_root_node_from_game(game=game.copy())
mcts = MCTS(root_node=root_node, rng=rng)

import cProfile
import pstats

with cProfile.Profile() as pr:
    mcts.run(n_iters=int(1e5))

stats = pstats.Stats(pr)
stats.sort_stats('cumulative')
stats.print_stats(20)

         31110087 function calls (31110080 primitive calls) in 27.568 seconds

   Ordered by: cumulative time
   List reduced from 123 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
   100000    0.190    0.000   28.005    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:34(_run_iteration)
        7    0.029    0.004   25.938    3.705 C:\Users\Samue\miniconda3\Lib\selectors.py:310(select)
    99765    5.150    0.000   18.576    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:105(_simulation)
  1506707    1.318    0.000    7.877    0.000 c:\Users\Samue\gitrepos\connect_four\src\game\connect_four.py:114(make_move)
   100000    0.329    0.000    4.741    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:75(_selection)
    99765    1.071    0.000    4.351    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:91(_expansion)
   606532    2.821    0.000    4

In [6]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    mcts.run(n_iters=int(1e4))

stats = pstats.Stats(pr)
stats.sort_stats('cumulative')
stats.print_stats(20)  # top 20 bottlenecks

         3152200 function calls in 2.785 seconds

   Ordered by: cumulative time
   List reduced from 95 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    10000    0.019    0.000    2.800    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:34(_run_iteration)
        1    0.000    0.000    2.652    2.652 C:\Users\Samue\miniconda3\Lib\asyncio\base_events.py:1962(_run_once)
        1    0.003    0.003    2.652    2.652 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:131(run)
     9983    0.504    0.000    1.792    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:105(_simulation)
   144288    0.131    0.000    0.756    0.000 c:\Users\Samue\gitrepos\connect_four\src\game\connect_four.py:114(make_move)
    10000    0.039    0.000    0.582    0.000 c:\Users\Samue\gitrepos\connect_four\src\mcts\mcts_algorithm.py:75(_selection)
    75198    0.345    0.000    0.534    0.000 c:\Users